# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
# =========================================================
# ML-06 — Signal Audit
# 1. Distributions
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the same dataset used in ML-05
DATA_PATH = r"D:\Internship\Week1\Task1-Week1\work\outputs\baseline_action_score.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Key numeric fields for distribution checks
distribution_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

# Keep only fields that exist
distribution_features = [
    col for col in distribution_features
    if col in df.columns
]

print("\nDistribution summary:")
display(
    df[distribution_features].describe().T[
        ["count", "mean", "50%", "std", "min", "max"]
    ]
)

# Skewness check
skewness = df[distribution_features].skew().sort_values(
    key=lambda x: x.abs(),
    ascending=False
)

print("\nSkewness:")
display(skewness.to_frame("skewness"))

# Flag strongly skewed fields
heavy_tail_features = skewness[
    skewness.abs() > 1
].index.tolist()

print("\nFields with notable skew / possible heavy tails:")
print(heavy_tail_features)

Dataset shape: (30000, 47)

Distribution summary:


,count,mean,50%,std,min,max
search_volume,27532.0,158.882391,10.00,1518.270825,0.0,74000.00
competition,27532.0,0.146954,0.00,0.285241,0.0,1.00
cpc,27532.0,0.485342,0.00,2.101560,0.0,100.36
word_count,22301.0,3107.760325,2877.00,1452.382598,8.0,9546.00
impressions_90d,30000.0,5200.366300,731.00,16838.019547,1.0,517715.00
clicks_90d,30000.0,16.097333,1.00,75.076958,0.0,4178.00
sessions_90d,30000.0,37.066633,7.00,107.069131,1.0,4345.00
content_age_days,30000.0,256.167800,236.00,132.707930,90.0,564.00
ctr,30000.0,0.510733,0.07,3.279162,0.0,100.00
avg_position,30000.0,16.342380,10.80,15.216790,0.0,245.00



Skewness:


,skewness
trend_pct,56.764773
search_volume,26.016250
ai_traffic_pct,18.432064
clicks_90d,18.345790
ctr,17.444252
cpc,13.737601
sessions_90d,12.126852
impressions_90d,11.384919
engagement_rate,7.222669
scroll_rate,2.499859



Fields with notable skew / possible heavy tails:
['trend_pct', 'search_volume', 'ai_traffic_pct', 'clicks_90d', 'ctr', 'cpc', 'sessions_90d', 'impressions_90d', 'engagement_rate', 'scroll_rate', 'competition', 'avg_position']


I checked the skewness of the main numeric features before testing the signals.

Several features are strongly right-skewed. The strongest skew was observed in `trend_pct` (56.76), followed by `search_volume` (26.02), `ai_traffic_pct` (18.43), `clicks_90d` (18.35), and `ctr` (17.44).

This means a small number of content items have unusually large values compared with most rows. These heavy tails should be considered when interpreting averages and signal relationships.

I therefore use medians, grouped comparisons, and directional checks rather than relying only on means.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

I tested three directional signals using grouped comparisons.

Signal #1 tests whether declining trend is associated with lower recent sessions.

Signal #2 tests whether higher engagement rate is associated with higher recent sessions.

Signal #3 tests whether stronger search demand is associated with higher recent impressions.

The tests are descriptive and directional. They do not prove causation.

In [3]:
# =========================================================
# Signal Test #1
# Does declining trend relate to lower recent sessions?
# =========================================================

signal1 = df[
    ["trend_direction", "sessions_last_30d"]
].dropna()

signal1_summary = (
    signal1
    .groupby("trend_direction")["sessions_last_30d"]
    .agg(["count", "median", "mean"])
    .sort_values("median")
)

display(signal1_summary)

print(
    "\nSignal #1 checks whether trend direction is associated "
    "with recent session levels."
)

,count,median,mean
trend_direction,,,
flat,1152,0.0,0.828125
new,2236,1.0,5.896243
down,16262,2.0,11.823699
up,4388,3.0,14.390155
stable,5962,8.0,25.808286



Signal #1 checks whether trend direction is associated with recent session levels.


### Signal #1 verdict: MIXED

The grouped results show a directional relationship between trend direction and recent sessions, but the groups are not sufficient to establish a strong or causal relationship.

I therefore treat this signal as MIXED rather than CONFIRMED.

In [4]:
# =========================================================
# Signal Test #2
# Does higher engagement relate to higher recent sessions?
# =========================================================

signal2 = df[
    ["engagement_rate", "sessions_last_30d"]
].dropna()

signal2["engagement_group"] = pd.qcut(
    signal2["engagement_rate"],
    q=4,
    duplicates="drop"
)

signal2_summary = (
    signal2
    .groupby("engagement_group", observed=True)["sessions_last_30d"]
    .agg(["count", "median", "mean"])
)

display(signal2_summary)

print(
    "\nSignal #2 compares recent sessions across engagement-rate quartiles."
)

,count,median,mean
engagement_group,,,
"(-0.001, 1.35]",22511,1.0,8.389765
"(1.35, 100.0]",7489,13.0,31.321405



Signal #2 compares recent sessions across engagement-rate quartiles.



### Signal #2 verdict: MIXED

The grouped comparison provides directional evidence about whether higher engagement is associated with higher recent sessions, but the relationship is not strong enough to treat as a confirmed rule.

The result should be used as decision-support rather than as a causal claim.

In [5]:
# =========================================================
# Signal Test #3
# Does higher search volume relate to higher impressions?
# =========================================================

signal3 = df[
    ["search_volume", "impressions_90d"]
].dropna()

signal3["search_volume_group"] = pd.qcut(
    signal3["search_volume"],
    q=4,
    duplicates="drop"
)

signal3_summary = (
    signal3
    .groupby("search_volume_group", observed=True)["impressions_90d"]
    .agg(["count", "median", "mean"])
)

display(signal3_summary)

print(
    "\nSignal #3 compares impressions across search-volume quartiles."
)

,count,median,mean
search_volume_group,,,
"(-0.001, 10.0]",18392,929.0,5547.994182
"(10.0, 20.0]",2290,1006.5,5690.318341
"(20.0, 74000.0]",6850,842.5,5796.309635



Signal #3 compares impressions across search-volume quartiles.


### Signal #3 verdict: MIXED

The grouped results provide directional evidence about the relationship between search volume and impressions, but the heavy skew in search volume means the relationship should be interpreted cautiously.

The signal is therefore treated as MIXED rather than as a guaranteed rule.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

One of the baseline flags uses declining trend as part of its decision logic.

I therefore tested whether the observed trend direction is associated with recent performance. This checks whether the data provides support for using trend direction as a decision signal.

In [6]:
# =========================================================
# Section 3 - Flag-linked test
# =========================================================

flag_test = df[
    [
        "trend_direction",
        "trend_pct",
        "sessions_last_30d",
        "sessions_prev_30d"
    ]
].dropna()

flag_test["session_change_pct"] = np.where(
    flag_test["sessions_prev_30d"] > 0,
    (
        (flag_test["sessions_last_30d"] -
         flag_test["sessions_prev_30d"])
        / flag_test["sessions_prev_30d"]
    ) * 100,
    np.nan
)

flag_summary = (
    flag_test
    .groupby("trend_direction")
    .agg(
        rows=("trend_pct", "size"),
        median_trend_pct=("trend_pct", "median"),
        median_session_change_pct=("session_change_pct", "median")
    )
    .sort_values("median_trend_pct")
)

display(flag_summary)

print(
    "\nThis test checks whether the trend-direction flag "
    "is directionally consistent with recent session change."
)

,rows,median_trend_pct,median_session_change_pct
trend_direction,,,
down,16262,-55.60,0.000000
stable,5962,-3.80,32.623792
up,4388,62.55,39.286011



This test checks whether the trend-direction flag is directionally consistent with recent session change.


### Flag-linked verdict: MIXED

The data provides some directional support for using trend direction as a signal, but the relationship is not strong enough to treat the flag as universally reliable.

The flag is therefore better viewed as decision-support rather than a standalone rule.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit shows that several content signals have heavy-tailed distributions, so extreme values can strongly affect averages.

The tested signals provide useful directional information, but they should not be treated as causal or guaranteed rules. A content team can use these signals to prioritize review, while keeping human judgment for the final action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.